# Quality Gate — 8-class document quality classifier

Trains `quality_gate` (replaces the old 3-class presentation attack model).

**Classes:** screen / printout / selfie / back / garbage / good_front / partial / blurry

**Workflow:**
1. Export labels from Label Studio as JSON → upload to `/content/label_studio_export.json`
2. Run Section 2 to convert labels → training dataset (downloads images from S3)
3. Run Section 3 to train (40 epochs, ~20 min on T4)
4. Run Section 4 to save weights to Drive

In [ ]:
# Quick setup — mounts Drive, pulls repo, rebuilds splits (no S3 re-download)
import os, sys, subprocess, importlib
from pathlib import Path

GITHUB_TOKEN = ""  # only if repo is private

REPO = Path("/content/id-forensics-model")
os.chdir("/content")
if not REPO.is_dir():
    user, repo = "ansisvaisla", "id-forensics-model"
    url = (
        f"https://{GITHUB_TOKEN}@github.com/{user}/{repo}.git"
        if GITHUB_TOKEN else f"https://github.com/{user}/{repo}.git"
    )
    subprocess.run(["git", "clone", url, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "scripts"))
import colab_bootstrap as cb
importlib.reload(cb)

# sync_images=False → images already on Drive from first-time setup
cb.setup(github_token=GITHUB_TOKEN, sync_images=False)

In [ ]:
EPOCHS = 40
BATCH = 32
DEVICE = "cuda"

# Path to the Label Studio JSON export you uploaded to Colab
LS_EXPORT = "/content/label_studio_export.json"

In [ ]:
# Section 2 — Convert Label Studio export → training dataset
# Downloads labelled images from S3 into data/yolo/screen/{train,val}/
!cd /content/id-forensics-model && python scripts/convert_labels_ls_to_quality_gate.py \
    --input {LS_EXPORT} \
    --out data/yolo/screen \
    --val-ratio 0.2

In [ ]:
!cd /content/id-forensics-model && python scripts/training/train_stage2_screen.py --device {DEVICE} --epochs {EPOCHS} --batch {BATCH}

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/id-forensics-model/scripts")
import colab_bootstrap as cb
importlib.reload(cb)

cb.save_weights("stage2")